# 4_MODEL — Model Validation & Diagnostics

Fits QRF, GBM (quantile + conformal), and Similarity on the IHFC reference parquet  
using the parameter files produced by `3b_MODEL_SWEEPS`.  
Saves trained model files and writes full diagnostic metrics.

**Sections**
1. Configuration & paths  
2. Load reference data  
3. Train/test split & scaler  
4. QRF — fit, save, metrics  
5. GBM — fit (quantile + point), conformal calibration, save, metrics  
6. Similarity — correction curve, save, metrics  
7. Metric summary table  
8. Diagnostic plots  


## 1 · Configuration & paths

In [ ]:
import sys, json, pickle, warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from quantile_forest import RandomForestQuantileRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
warnings.filterwarnings('ignore')

# ── user-adjustable ─────────────────────────────────────────────────────
local_data   = Path('data')
param_dir    = Path('output/sweeps')
model_dir    = Path('output/models')
fig_dir      = Path('output/figures/4_MODEL')
TEST_FRAC    = 0.20
RANDOM_SEED  = 42
TARGET_COL   = 'q'          # heat-flow column name in parquet
Q_CLIP_MIN   = 0.0
QUANTILES    = [0.05, 0.25, 0.50, 0.75, 0.95]
HIST_BIN_W   = 0.010
CONFORMAL_ALPHA = 0.10
# ── end user section ────────────────────────────────────────────────────

model_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

with open(param_dir / 'qrf_params.json') as f:   QP = json.load(f)
with open(param_dir / 'gbm_params.json') as f:   GP = json.load(f)
with open(param_dir / 'sim_best_params.json') as f: SP = json.load(f)

from config import obs_model, q_clip_max
print(f'Features  : {len(obs_model)}')
print(f'q_clip_max: {q_clip_max} W/m²')
print(f'Test frac : {TEST_FRAC}')


Features  : 22
q_clip_max: 0.25 W/m²
Test frac : 0.2


## 2 · Load reference data

In [10]:
parquet_ref = local_data / 'IHFC_obs.parquet'
df = pd.read_parquet(parquet_ref)
print(f'Raw rows: {len(df)}')

print(list(df))

print(obs_model)   

Raw rows: 30848
['lon', 'lat', 'q', 'elevation', 'Domain', 'Quality_Code', 'MOHO_GRAV', 'MOHO_GRAV_U', 'MOHO', 'MOHO_U', 'DEM', 'LAB', 'CTD', 'EMAG2_LOG', 'FREE_AIR', 'BOUGUER', 'SI', 'GEOID', 'REVEAL_S50', 'REVEAL_S80', 'REVEAL_S90', 'REVEAL_S100', 'REVEAL_S140', 'REVEAL_S200', 'REVEAL_S220', 'REVEAL_P40', 'REVEAL_P60', 'REVEAL_P120', 'REVEAL_P150', 'REVEAL_P180', 'REVEAL_P210', 'LITH_RHO', 'CRUST_RHO', 'GLiM', 'TC1', 'GPRV', 'REG', 'SEDIMENT', 'MAG_SEIS_MOHO', 'REVEAL_S_DIFF_200_220', 'REVEAL_P_DIFF_40_60', 'VOLC_DIST', 'weight']
['MOHO_GRAV', 'MOHO', 'DEM', 'LAB', 'FREE_AIR', 'BOUGUER', 'SI', 'GEOID', 'REVEAL_S80', 'REVEAL_S90', 'REVEAL_P180', 'REVEAL_P150', 'LITH_RHO', 'CRUST_RHO', 'MAG_SEIS_MOHO', 'SEDIMENT', 'S_DIFF_200_220', 'P_DIFF_40_60', 'VP_40_VS_50', 'VP_120_VS_140', 'CTD', 'EMAG2_LOG']


In [11]:

print(f'After clip & drop-NaN: {len(df)} rows')

X_all = df[obs_model].values.astype(np.float32)
y_all = df[TARGET_COL].values.astype(np.float32)

# weights from 1_Import (ref.df["weight"]).
# Down-weights over-represented regions so dense clusters don't dominate.
# Falls back to uniform weights if column absent.
if 'weight' in df.columns:
    w_all = df['weight'].values.astype(np.float32)
    w_all = w_all / w_all.mean()   # normalise so mean weight = 1
    print(f'Weights loaded  min={w_all.min():.4f}  max={w_all.max():.4f}  mean={w_all.mean():.4f}')
else:
    w_all = np.ones(len(y_all), dtype=np.float32)
    print('WARNING: no weight column found — using uniform weights')


After clip & drop-NaN: 30848 rows


KeyError: "['S_DIFF_200_220', 'P_DIFF_40_60', 'VP_40_VS_50', 'VP_120_VS_140'] not in index"

## 3 · Train/test split & scaler

In [ ]:
# train_test_split with weights — X, y, w split consistently
X_tr_raw, X_te_raw, y_tr, y_te, w_tr, w_te = train_test_split(
    X_all, y_all, w_all, test_size=TEST_FRAC, random_state=RANDOM_SEED)

# Scaler fitted on TRAIN only — no leakage
scaler = StandardScaler().fit(X_tr_raw)
X_tr   = scaler.transform(X_tr_raw)
X_te   = scaler.transform(X_te_raw)

# σ array for Similarity kernel
sigma_arr = np.array([SP['SIM_SIGMAS'].get(f, 1.0) for f in obs_model], dtype=np.float32)

print(f'Train: {len(y_tr)}   Test: {len(y_te)}')
print(f'sigma placeholders: {SP.get("sigma_placeholders", [])}')


## 4 · Quantile Random Forest

QRF uses the `quantile_forest` library, which stores leaf-node sample  
indices so it can return any quantile (or the full empirical distribution)  
at prediction time — no need to re-train for different quantile levels.

In [ ]:
qrf_path = model_dir / 'qrf_model.pkl'
if qrf_path.exists():
    print('QRF: loading saved model')
    with open(qrf_path, 'rb') as f:
        qrf = pickle.load(f)
else:
    print('QRF: training...')
    qrf = RandomForestQuantileRegressor(
        n_estimators     = QP['n_estimators'],
        max_depth        = QP['max_depth'] if QP['max_depth'] != 'None' else None,
        min_samples_leaf = QP['min_samples_leaf'],
        max_features     = QP['max_features'],
        n_jobs           = -1,
        random_state     = RANDOM_SEED,
    )
    qrf.fit(X_tr, y_tr, sample_weight=w_tr)
    with open(qrf_path, 'wb') as f:
        pickle.dump(qrf, f)
    print(f'QRF saved → {qrf_path}')

# ── predict quantiles on test set ────────────────────────────────────────
# Predict at 5 named quantiles + a dense grid for mean/std/skew
QUANTILES_DENSE = np.linspace(0.02, 0.98, 49).tolist()
qrf_preds  = qrf.predict(X_te, quantiles=QUANTILES)         # (n, 5)
qrf_dense  = qrf.predict(X_te, quantiles=QUANTILES_DENSE)   # (n, 49)
qrf_q50    = qrf_preds[:, QUANTILES.index(0.50)]
# mean and std from the dense quantile grid (unweighted approx of empirical dist)
qrf_mean   = qrf_dense.mean(axis=1)
qrf_std    = qrf_dense.std(axis=1)

# ── conformal prediction ─────────────────────────────────────────────────
# Split conformal: use a held-back slice of train as calibration set.
# We take cal_frac of the original training set.
cal_frac = QP.get('cal_frac', 0.10)
n_cal    = int(len(X_tr) * cal_frac)
X_cal, y_cal, w_cal = X_tr[-n_cal:], y_tr[-n_cal:], w_tr[-n_cal:]
X_tr_fit, y_tr_fit, w_tr_fit = X_tr[:-n_cal], y_tr[:-n_cal], w_tr[:-n_cal]
# Non-conformity scores on cal set
# passing both quantiles together returns (n,2); single-element list returns 1-D
cal_preds = qrf.predict(X_cal, quantiles=[0.05, 0.95])   # (n_cal, 2)
cal_q05, cal_q95 = cal_preds[:, 0], cal_preds[:, 1]
scores   = np.maximum(cal_q05 - y_cal, y_cal - cal_q95)
qhat     = np.quantile(scores, (1 - CONFORMAL_ALPHA) * (1 + 1/len(scores)))
qrf_conf_lo  = qrf_preds[:, 0] - qhat
qrf_conf_hi  = qrf_preds[:, -1] + qhat
qrf_picp_conf = np.mean((y_te >= qrf_conf_lo) & (y_te <= qrf_conf_hi))

# ── metrics ──────────────────────────────────────────────────────────────
qrf_r2   = r2_score(y_te, qrf_q50)
qrf_rmse = np.sqrt(mean_squared_error(y_te, qrf_q50))
qrf_mae  = mean_absolute_error(y_te, qrf_q50)
qrf_pi90_w = np.mean(qrf_preds[:,-1] - qrf_preds[:,0])   # raw PI90 width
qrf_picp_raw = np.mean((y_te>=qrf_preds[:,0])&(y_te<=qrf_preds[:,-1]))
print(f'QRF  R²={qrf_r2:.4f}  RMSE={qrf_rmse*1000:.2f} mW/m²  MAE={qrf_mae*1000:.2f}  '
      f'PI90_coverage={qrf_picp_raw:.3f}  Conformal_coverage={qrf_picp_conf:.3f}')

## 5 · Gradient Boosting Regression

Three separate `HistGradientBoostingRegressor` models are trained at  
quantile losses Q05 / Q50 / Q95. Conformal calibration is applied  
post-hoc to produce coverage-guaranteed prediction intervals without  
extra training runs.

In [ ]:
gbm_paths = {q: model_dir / f'gbm_q{int(q*100):02d}_model.pkl'
             for q in GP['quantile_losses']}

gbm_models = {}
for q, path in gbm_paths.items():
    if path.exists():
        print(f'GBM Q{int(q*100):02d}: loading saved model')
        with open(path, 'rb') as f:
            gbm_models[q] = pickle.load(f)
    else:
        print(f'GBM Q{int(q*100):02d}: training...')
        loss = 'quantile' if q != 0.5 else 'squared_error'
        params = dict(
            loss              = loss,
            quantile          = q if q != 0.5 else None,
            max_iter          = GP['max_iter'],
            max_depth         = GP['max_depth'],
            learning_rate     = GP['learning_rate'],
            min_samples_leaf  = GP['min_samples_leaf'],
            max_features      = GP['max_features'],
            l2_regularization = GP['l2_regularization'],
            random_state      = RANDOM_SEED,
        )
        if q == 0.5:
            params.pop('quantile')
        gbm_models[q] = HistGradientBoostingRegressor(**params).fit(X_tr, y_tr, sample_weight=w_tr)
        with open(path, 'wb') as f:
            pickle.dump(gbm_models[q], f)
        print(f'  saved → {path}')

# ── test-set predictions ──────────────────────────────────────────────────
gbm_q05   = gbm_models[0.05].predict(X_te)
gbm_q50   = gbm_models[0.50].predict(X_te)
gbm_q95   = gbm_models[0.95].predict(X_te)

# ── conformal calibration ─────────────────────────────────────────────────
cal_05 = gbm_models[0.05].predict(X_cal)
cal_95 = gbm_models[0.95].predict(X_cal)
scores_gbm = np.maximum(cal_05 - y_cal, y_cal - cal_95)
qhat_gbm   = np.quantile(scores_gbm, (1-CONFORMAL_ALPHA)*(1+1/len(scores_gbm)))
gbm_conf_lo  = gbm_q05 - qhat_gbm
gbm_conf_hi  = gbm_q95 + qhat_gbm
gbm_picp_conf = np.mean((y_te>=gbm_conf_lo)&(y_te<=gbm_conf_hi))

# ── metrics ───────────────────────────────────────────────────────────────
gbm_r2   = r2_score(y_te, gbm_q50)
gbm_rmse = np.sqrt(mean_squared_error(y_te, gbm_q50))
gbm_mae  = mean_absolute_error(y_te, gbm_q50)
gbm_pi90_w = np.mean(gbm_q95 - gbm_q05)
gbm_picp_raw = np.mean((y_te>=gbm_q05)&(y_te<=gbm_q95))
print(f'GBM  R²={gbm_r2:.4f}  RMSE={gbm_rmse*1000:.2f} mW/m²  MAE={gbm_mae*1000:.2f}  '
      f'PI90_coverage={gbm_picp_raw:.3f}  Conformal_coverage={gbm_picp_conf:.3f}')

## 6 · Similarity — correction curve

The kernel-weighted mean is mathematically biased toward the centre of the  
reference Q distribution (analogous to regression dilution). We correct this  
by fitting a monotone spline that maps predicted quantiles back onto the  
empirical quantiles of the reference distribution — effectively stretching  
the predicted distribution to match the reference range.  
This is computed from the **training set only** (no leakage).

In [ ]:
from scipy.interpolate import PchipInterpolator

def similarity_predict(X_tr_scaled, y_ref, w_ref, X_target_scaled, sigma_arr, K,
                       batch=512):
    """Batched similarity kernel.

    Inputs are StandardScaler-normalised (mean=0, std=1).
    sigma_arr applies per-feature bandwidth on top of that.
    NaN rows in X_target are returned as NaN in all outputs.

    Returns array (n_target, 4): [q_mean, q_median, q_std, N_eff]
    """
    sort_idx = np.argsort(y_ref)
    y_s = y_ref[sort_idx]
    w_ref_s = w_ref[sort_idx] if w_ref is not None else None

    results = np.full((len(X_target_scaled), 4), np.nan, dtype=np.float64)

    for i in range(0, len(X_target_scaled), batch):
        Xb = X_target_scaled[i:i+batch]                      # (B, F)

        # mask rows that contain any NaN
        valid = ~np.isnan(Xb).any(axis=1)                    # (B,)
        if not valid.any():
            continue
        Xv = Xb[valid]                                        # (V, F)

        diff = (Xv[:, None, :] - X_tr_scaled[None, :, :]) / sigma_arr  # (V, R, F)
        S    = np.exp(-0.5 * np.nanmean(diff**2, axis=2))               # (V, R)
        S_K  = S ** K

        # apply reference density weights before normalising
        if w_ref is not None:
            S_K = S_K * w_ref[None, :]
        denom = S_K.sum(axis=1, keepdims=True)
        # if all similarities collapse to 0 for a row, leave as NaN
        ok = (denom[:, 0] > 1e-30)
        w  = np.where(ok[:, None], S_K / (denom + 1e-30), np.nan)       # (V, R)

        q_mean = np.nansum(w * y_ref, axis=1)
        q_std  = np.sqrt(np.nansum(w * (y_ref - q_mean[:, None])**2, axis=1))

        # weighted median
        w_s    = w[:, sort_idx]
        cumw   = np.nancumsum(w_s, axis=1)
        tot    = cumw[:, -1]
        med_idx = np.argmax(cumw >= (tot[:, None] * 0.5), axis=1)
        q_med  = y_s[med_idx]

        N_eff  = 1.0 / np.nansum(w**2, axis=1)

        out = np.column_stack([q_mean, q_med, q_std, N_eff])
        out[~ok] = np.nan
        results[i:i+batch][valid] = out

    return results


K         = SP['SIM_BEST_K']
# IMPORTANT: pass X_tr / X_te as-is from StandardScaler (no extra /sigma_arr here).
# sigma_arr division happens once, inside similarity_predict.
print('Computing Similarity on test set...')
sim_te = similarity_predict(X_tr, y_tr, w_tr, X_te, sigma_arr, K)
sim_q_mean, sim_q_med, sim_q_std, sim_N_eff = sim_te.T

nan_frac = np.isnan(sim_q_mean).mean()
print(f'  NaN fraction in sim_q_mean: {nan_frac:.3%}')

# ── correction curve (train self-prediction — leave-in, intentional) ─────
# We predict train on train to learn the mean-reversion bias pattern.
print('Fitting correction curve on train predictions...')
sim_tr_out  = similarity_predict(X_tr, y_tr, w_tr, X_tr, sigma_arr, K)
sim_tr_mean = sim_tr_out[:, 0]

# Weighted percentiles (density-corrected reference distribution)
pctls = np.linspace(1, 99, 200)

def weighted_percentile(vals, weights, pcts):
    """Weighted quantile via sorted cumulative weights. Ignores NaNs."""
    mask = np.isfinite(vals) & np.isfinite(weights)
    vs   = vals[mask]; ws = weights[mask]
    idx  = np.argsort(vs)
    vs   = vs[idx]; ws = ws[idx]
    cumw = np.cumsum(ws) / ws.sum()
    return np.interp(pcts / 100.0, cumw, vs)

pred_pctls = weighted_percentile(sim_tr_mean, w_tr, pctls)
true_pctls = weighted_percentile(y_tr,        w_tr, pctls)

# Deduplicate for PchipInterpolator (requires strictly increasing x)
_, keep = np.unique(pred_pctls, return_index=True)
correction_spline = PchipInterpolator(pred_pctls[keep], true_pctls[keep], extrapolate=True)

# Apply correction; preserve NaNs
sim_q_mean_corr = np.where(np.isfinite(sim_q_mean),
                            correction_spline(sim_q_mean).astype(np.float32),
                            np.nan)
sim_q_med_corr  = np.where(np.isfinite(sim_q_med),
                            correction_spline(sim_q_med).astype(np.float32),
                            np.nan)

# Save artefacts for 5_TARGETS
corr_path = model_dir / 'sim_correction_spline.pkl'
with open(corr_path, 'wb') as f:
    pickle.dump({'pred_pctls': pred_pctls, 'true_pctls': true_pctls,
                 'scaler': scaler, 'sigma_arr': sigma_arr, 'K': K}, f)
print(f'Correction spline saved → {corr_path}')

# Metrics on non-NaN test rows only
valid_te = np.isfinite(sim_q_mean_corr) & np.isfinite(y_te)
sim_r2_raw  = r2_score(y_te[valid_te], sim_q_mean[valid_te])
sim_r2_corr = r2_score(y_te[valid_te], sim_q_mean_corr[valid_te])
sim_rmse    = np.sqrt(mean_squared_error(y_te[valid_te], sim_q_mean_corr[valid_te]))
sim_mae     = mean_absolute_error(y_te[valid_te], sim_q_mean_corr[valid_te])
print(f'SIM  R²(raw)={sim_r2_raw:.4f}  R²(corr)={sim_r2_corr:.4f}  '
      f'RMSE={sim_rmse*1000:.2f} mW/m²  MAE={sim_mae*1000:.2f} mW/m²')


## 7 · Shannon entropy & metric summary

In [ ]:
from scipy.stats import entropy as scipy_entropy

def empirical_entropy(vals, bin_width=HIST_BIN_W):
    """Shannon entropy (nats) of an empirical distribution."""
    bins = np.arange(Q_CLIP_MIN, q_clip_max + bin_width, bin_width)
    counts, _ = np.histogram(vals, bins=bins)
    probs = counts / counts.sum()
    return scipy_entropy(probs[probs > 0])

def pearson_bias(y_true, y_pred):
    return np.mean(y_pred - y_true)

H_obs  = empirical_entropy(y_te)
H_qrf  = empirical_entropy(qrf_q50)
H_gbm  = empirical_entropy(gbm_q50)
H_sim  = empirical_entropy(sim_q_mean_corr)

metrics = pd.DataFrame({
    'Method'         : ['QRF (Q50)','GBM (Q50)','Similarity (corrected)'],
    'R²'             : [qrf_r2, gbm_r2, sim_r2_corr],
    'RMSE [mW/m²]'   : [qrf_rmse*1e3, gbm_rmse*1e3, sim_rmse*1e3],
    'MAE [mW/m²]'    : [qrf_mae*1e3, gbm_mae*1e3, sim_mae*1e3],
    'Bias [mW/m²]'   : [pearson_bias(y_te,qrf_q50)*1e3,
                        pearson_bias(y_te,gbm_q50)*1e3,
                        pearson_bias(y_te,sim_q_mean_corr)*1e3],
    'PI90 coverage'  : [qrf_picp_conf, gbm_picp_conf, np.nan],
    'PI90 width [mW]': [qrf_pi90_w*1e3, gbm_pi90_w*1e3, np.nan],
    'Shannon H [nat]': [H_qrf, H_gbm, H_sim],
})
metrics = metrics.round(3)
print(metrics.to_string(index=False))
metrics.to_csv(model_dir / 'model_metrics.csv', index=False)
print(f'\nObserved entropy: {H_obs:.3f} nat')

## 8 · Diagnostic plots

In [ ]:
# ── helper: 2D histogram + 1:1 + residuals ───────────────────────────────
def scatter_panel(ax_scat, ax_res, y_true, y_pred, label, color):
    lim   = (Q_CLIP_MIN * 1e3, q_clip_max * 1e3)
    yt, yp = y_true * 1e3, y_pred * 1e3

    # 2D histogram background
    bins  = np.linspace(lim[0], lim[1], 80)
    h, xe, ye = np.histogram2d(yt, yp, bins=bins)
    # log-scale counts for visibility; mask zeros
    h = np.ma.masked_where(h == 0, h)
    ax_scat.pcolormesh(xe, ye, h.T, cmap='Blues', norm=plt.matplotlib.colors.LogNorm(),
                       rasterized=True, zorder=0)

    # marginal density lines (kernel-smoothed 1D projections on each axis)
    ax_scat.plot([lim[0], lim[1]], [lim[0], lim[1]], 'k--', lw=0.9, zorder=2, label='1:1')

    # R² and regression line
    r2  = r2_score(y_true, y_pred)
    m, b = np.polyfit(yt, yp, 1)
    xfit = np.array(lim)
    ax_scat.plot(xfit, m * xfit + b, color=color, lw=1.2, zorder=3,
                 label=f'fit  R²={r2:.3f}')
    ax_scat.set_xlim(lim); ax_scat.set_ylim(lim)
    ax_scat.set_xlabel('Observed [mW/m²]')
    ax_scat.set_ylabel('Predicted [mW/m²]')
    ax_scat.set_title(label)
    ax_scat.legend(fontsize=8, loc='upper left')

    # residual histogram
    resid = yp - yt
    ax_res.hist(resid, bins=60, color=color, alpha=0.75, edgecolor='none')
    ax_res.axvline(0, color='k', lw=0.9)
    ax_res.set_xlabel('Residual [mW/m²]')
    ax_res.set_ylabel('Count')
    ax_res.set_title(f'Bias={np.mean(resid):.2f}  σ={np.std(resid):.2f} mW/m²')


# filter valid rows for similarity (may contain NaN)
sim_valid = np.isfinite(sim_q_mean_corr) & np.isfinite(y_te)

fig, axes = plt.subplots(3, 2, figsize=(12, 14))
scatter_panel(axes[0,0], axes[0,1], y_te,             qrf_q50,                  'QRF Q50',          '#2196F3')
scatter_panel(axes[1,0], axes[1,1], y_te,             gbm_q50,                  'GBM Q50',          '#4CAF50')
scatter_panel(axes[2,0], axes[2,1], y_te[sim_valid],  sim_q_mean_corr[sim_valid],'Similarity (corr)','#FF5722')
fig.suptitle('Model diagnostics — held-out test set (20%)', fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(fig_dir / 'scatter_residuals.png', dpi=150, bbox_inches='tight')
plt.show()


# ── correction curve visualisation ────────────────────────────────────────
fig2, ax2 = plt.subplots(1, 2, figsize=(11, 4))
ax2[0].plot(pred_pctls * 1e3, true_pctls * 1e3, color='#FF5722', lw=1.5, label='Correction spline')
ax2[0].plot([0, q_clip_max*1e3], [0, q_clip_max*1e3], 'k--', lw=0.8, label='1:1')
ax2[0].set_xlabel('Predicted Q [mW/m²]')
ax2[0].set_ylabel('Corrected Q [mW/m²]')
ax2[0].set_title('Similarity mean-reversion correction')
ax2[0].legend()
bins50 = np.linspace(0, q_clip_max * 1e3, 51)
ax2[1].hist(y_te * 1e3,                        bins=bins50, alpha=0.5, label='Observed',       color='#333')
ax2[1].hist(sim_q_mean[sim_valid] * 1e3,        bins=bins50, alpha=0.5, label='Sim raw mean',   color='#FF5722')
ax2[1].hist(sim_q_mean_corr[sim_valid] * 1e3,   bins=bins50, alpha=0.5, label='Sim corrected',  color='#E91E63')
ax2[1].set_xlabel('Q [mW/m²]')
ax2[1].set_title('Distribution shift')
ax2[1].legend(fontsize=8)
fig2.tight_layout()
fig2.savefig(fig_dir / 'sim_correction.png', dpi=150, bbox_inches='tight')
plt.show()


# ── PI width vs observed Q — 2D histogram ────────────────────────────────
fig3, ax3 = plt.subplots(1, 2, figsize=(11, 4))
for ax, lo, hi, lab, col in [
    (ax3[0], qrf_conf_lo, qrf_conf_hi, 'QRF conformal PI90',  '#2196F3'),
    (ax3[1], gbm_conf_lo, gbm_conf_hi, 'GBM conformal PI90',  '#4CAF50'),
]:
    width  = (hi - lo) * 1e3
    xbins  = np.linspace(0, q_clip_max * 1e3, 60)
    ybins  = np.linspace(0, np.percentile(width, 99), 60)
    h2, xe, ye = np.histogram2d(y_te * 1e3, width, bins=[xbins, ybins])
    h2 = np.ma.masked_where(h2 == 0, h2)
    ax.pcolormesh(xe, ye, h2.T, cmap='Greens' if col == '#4CAF50' else 'Blues',
                  norm=plt.matplotlib.colors.LogNorm(), rasterized=True)
    ax.axhline(np.mean(width), color=col, lw=1.2, linestyle='--',
               label=f'mean={np.mean(width):.1f} mW')
    ax.set_xlabel('Observed Q [mW/m²]')
    ax.set_ylabel('PI width [mW/m²]')
    ax.set_title(lab)
    ax.legend(fontsize=8)
fig3.tight_layout()
fig3.savefig(fig_dir / 'PI_width.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nAll figures saved to', fig_dir)


## 9 · Quantile-mapping calibration (spread restoration)

QRF and GBM minimise squared-error loss, so their point predictions are
conditional means — mathematically guaranteed to have less variance than the
target distribution (*regression dilution*). The correction fits a monotone
spline that maps the predicted quantile distribution back onto the weighted
reference distribution, identical to the Similarity correction in §6.
All splines are fitted on **train predictions only** (no leakage).
Corrected outputs are saved to the correction pickle for use in 5_TARGETS.

In [ ]:
from scipy.interpolate import PchipInterpolator
from scipy.ndimage import uniform_filter1d
import matplotlib.ticker as mticker

# ── shared weighted-percentile helper (may already exist from §6) ─────────
def weighted_percentile(vals, weights, pcts):
    mask = np.isfinite(vals) & np.isfinite(weights)
    vs, ws = vals[mask], weights[mask]
    idx  = np.argsort(vs); vs = vs[idx]; ws = ws[idx]
    cumw = np.cumsum(ws) / ws.sum()
    return np.interp(pcts / 100.0, cumw, vs)

pctls = np.linspace(1, 99, 200)
true_pctls_w = weighted_percentile(y_tr, w_tr, pctls)


def fit_correction(pred_tr_vals, weights, label):
    """Fit a weighted quantile-mapping spline on training predictions."""
    pp = weighted_percentile(pred_tr_vals, weights, pctls)
    _, keep = np.unique(pp, return_index=True)
    spline = PchipInterpolator(pp[keep], true_pctls_w[keep], extrapolate=True)
    print(f'  {label}: spline knots={len(keep)}  '
          f'pred range [{pp.min()*1e3:.1f}, {pp.max()*1e3:.1f}] mW/m²')
    return spline, pp


def apply_corr(spline, vals):
    """Apply correction, preserving NaNs."""
    return np.where(np.isfinite(vals),
                    spline(vals).astype(np.float32), np.nan)


print('Fitting calibration splines on training predictions...')

# QRF — correct Q50 (median) train prediction
qrf_tr_q50  = qrf.predict(X_tr, quantiles=[0.50, 0.05, 0.95])
qrf_tr_med  = qrf_tr_q50[:, 0]
qrf_tr_q05  = qrf_tr_q50[:, 1]
qrf_tr_q95  = qrf_tr_q50[:, 2]

qrf_corr_med, qrf_pp_med = fit_correction(qrf_tr_med, w_tr, 'QRF Q50')
qrf_corr_q05, _          = fit_correction(qrf_tr_q05, w_tr, 'QRF Q05')
qrf_corr_q95, _          = fit_correction(qrf_tr_q95, w_tr, 'QRF Q95')

# GBM — correct Q50, Q05, Q95
gbm_tr_med = gbm_models[0.50].predict(X_tr)
gbm_tr_q05 = gbm_models[0.05].predict(X_tr)
gbm_tr_q95 = gbm_models[0.95].predict(X_tr)

gbm_corr_med, gbm_pp_med = fit_correction(gbm_tr_med, w_tr, 'GBM Q50')
gbm_corr_q05, _          = fit_correction(gbm_tr_q05, w_tr, 'GBM Q05')
gbm_corr_q95, _          = fit_correction(gbm_tr_q95, w_tr, 'GBM Q95')

# Apply to test set
qrf_q50_corr = apply_corr(qrf_corr_med, qrf_q50)
gbm_q50_corr = apply_corr(gbm_corr_med, gbm_q50)

# Also re-correct QRF/GBM PI bounds so widths stay consistent
qrf_conf_lo_corr = apply_corr(qrf_corr_q05, qrf_conf_lo)
qrf_conf_hi_corr = apply_corr(qrf_corr_q95, qrf_conf_hi)
gbm_conf_lo_corr = apply_corr(gbm_corr_q05, gbm_conf_lo)
gbm_conf_hi_corr = apply_corr(gbm_corr_q95, gbm_conf_hi)

# Save all correction splines into the artefact pickle
with open(model_dir / 'sim_correction_spline.pkl', 'rb') as f:
    artefacts = pickle.load(f)
artefacts.update({
    'qrf_corr_med': qrf_corr_med, 'qrf_corr_q05': qrf_corr_q05, 'qrf_corr_q95': qrf_corr_q95,
    'gbm_corr_med': gbm_corr_med, 'gbm_corr_q05': gbm_corr_q05, 'gbm_corr_q95': gbm_corr_q95,
})
with open(model_dir / 'sim_correction_spline.pkl', 'wb') as f:
    pickle.dump(artefacts, f)
print('All correction splines saved.')


# ─────────────────────────────────────────────────────────────────────────────
# Figure A: 2D histogram + linear + LOESS regression + 1:1
#           Before and after correction for all three methods
# ─────────────────────────────────────────────────────────────────────────────

def loess_line(x, y, frac=0.4, n_pts=80):
    """Simple LOESS smoother via local linear regression."""
    from sklearn.linear_model import LinearRegression
    xs = np.linspace(x.min(), x.max(), n_pts)
    ys = np.empty(n_pts)
    bw = frac * (x.max() - x.min())
    for k, xi in enumerate(xs):
        w = np.exp(-0.5 * ((x - xi) / bw)**2)
        if w.sum() < 1e-6:
            ys[k] = np.nan; continue
        lr = LinearRegression().fit(x[:, None], y, sample_weight=w)
        ys[k] = lr.predict([[xi]])[0]
    return xs, ys


def density_panel(ax, yt, yp, title, color, lim_mw):
    """2D histogram + regression lines + 1:1."""
    bins = np.linspace(lim_mw[0], lim_mw[1], 70)
    h, xe, ye = np.histogram2d(yt, yp, bins=bins)
    h = np.ma.masked_where(h == 0, h)
    ax.pcolormesh(xe, ye, h.T, cmap='Blues',
                  norm=plt.matplotlib.colors.LogNorm(), rasterized=True, zorder=0)

    # 1:1
    ax.plot(lim_mw, lim_mw, 'k--', lw=0.9, zorder=3, label='1:1')

    # linear regression
    m, b = np.polyfit(yt, yp, 1)
    xf = np.array(lim_mw)
    ax.plot(xf, m*xf+b, color=color, lw=1.4, zorder=4,
            label=f'Linear  slope={m:.2f}')

    # LOESS
    xs, ys = loess_line(yt, yp, frac=0.35)
    ax.plot(xs, ys, color=color, lw=1.4, ls=':', zorder=4, label='LOESS')

    r2  = r2_score(yt/1e3, yp/1e3)
    rmse = np.sqrt(mean_squared_error(yt/1e3, yp/1e3)) * 1e3
    ax.set_xlim(lim_mw); ax.set_ylim(lim_mw)
    ax.set_xlabel('Observed [mW/m²]', fontsize=8)
    ax.set_ylabel('Predicted [mW/m²]', fontsize=8)
    ax.set_title(f'{title}\nR²={r2:.3f}  RMSE={rmse:.1f} mW/m²', fontsize=8)
    ax.legend(fontsize=7, loc='upper left')
    return r2, rmse


sim_valid = np.isfinite(sim_q_mean_corr) & np.isfinite(y_te)
lim_mw = (Q_CLIP_MIN * 1e3, q_clip_max * 1e3)

pairs = [
    # (y_true, y_pred_raw,       y_pred_corr,       label,        color)
    (y_te,            qrf_q50,             qrf_q50_corr,        'QRF',        '#2196F3'),
    (y_te,            gbm_q50,             gbm_q50_corr,        'GBM',        '#4CAF50'),
    (y_te[sim_valid], sim_q_mean[sim_valid], sim_q_mean_corr[sim_valid], 'Similarity', '#FF5722'),
]

fig, axes = plt.subplots(3, 2, figsize=(11, 14))
metrics_before, metrics_after = [], []

for row, (yt, yp_raw, yp_corr, lab, col) in enumerate(pairs):
    yt_mw  = yt  * 1e3
    raw_mw = yp_raw  * 1e3
    cor_mw = yp_corr * 1e3
    r2_b, rmse_b = density_panel(axes[row,0], yt_mw, raw_mw,
                                  f'{lab} — raw',        col, lim_mw)
    r2_a, rmse_a = density_panel(axes[row,1], yt_mw, cor_mw,
                                  f'{lab} — calibrated', col, lim_mw)
    metrics_before.append({'Method': lab, 'R²': r2_b, 'RMSE': rmse_b})
    metrics_after.append( {'Method': lab, 'R²': r2_a, 'RMSE': rmse_a})

fig.suptitle('Calibration (spread restoration) — before vs after', fontsize=12)
fig.tight_layout()
fig.savefig(fig_dir / 'calibration_2dhist.png', dpi=150, bbox_inches='tight')
plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# Figure B: metric trade-off — R² and RMSE before/after for all three models
# ─────────────────────────────────────────────────────────────────────────────
df_b = pd.DataFrame(metrics_before).set_index('Method')
df_a = pd.DataFrame(metrics_after).set_index( 'Method')

methods = df_b.index.tolist()
x  = np.arange(len(methods))
w2 = 0.32
colors = ['#2196F3', '#4CAF50', '#FF5722']

fig4, (ax_r2, ax_rm) = plt.subplots(1, 2, figsize=(11, 4))

bars_b = ax_r2.bar(x - w2/2, df_b['R²'], width=w2, label='Raw',        color=colors, alpha=0.55)
bars_a = ax_r2.bar(x + w2/2, df_a['R²'], width=w2, label='Calibrated', color=colors, alpha=0.95)
ax_r2.set_xticks(x); ax_r2.set_xticklabels(methods)
ax_r2.set_ylabel('R²')
ax_r2.set_title('R² — calibration slightly reduces point-prediction skill')
ax_r2.legend(fontsize=8)
# annotate delta
for i, (rb, ra) in enumerate(zip(df_b['R²'], df_a['R²'])):
    ax_r2.annotate(f'Δ{(ra-rb):+.3f}',
                   xy=(x[i] + w2/2, ra + 0.005),
                   ha='center', fontsize=7, color='#333')

bars_b2 = ax_rm.bar(x - w2/2, df_b['RMSE'], width=w2, label='Raw',        color=colors, alpha=0.55)
bars_a2 = ax_rm.bar(x + w2/2, df_a['RMSE'], width=w2, label='Calibrated', color=colors, alpha=0.95)
ax_rm.set_xticks(x); ax_rm.set_xticklabels(methods)
ax_rm.set_ylabel('RMSE [mW/m²]')
ax_rm.set_title('RMSE — calibration increases RMSE (expected trade-off)')
ax_rm.legend(fontsize=8)
for i, (rb, ra) in enumerate(zip(df_b['RMSE'], df_a['RMSE'])):
    ax_rm.annotate(f'Δ{(ra-rb):+.1f}',
                   xy=(x[i] + w2/2, ra + 0.1),
                   ha='center', fontsize=7, color='#333')

fig4.suptitle(
    'Calibration trade-off: distributional fidelity ↑  vs  point-prediction RMSE ↑',
    fontsize=10)
fig4.tight_layout()
fig4.savefig(fig_dir / 'calibration_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

# Print summary table
print('\n── Calibration metric comparison ──────────────────────────────────')
print(f'{"Method":<14}  {"R² raw":>7}  {"R² cal":>7}  {"ΔR²":>7}  '
      f'{"RMSE raw":>9}  {"RMSE cal":>9}  {"ΔRMSE":>7}')
for lab in methods:
    rb, ra   = df_b.loc[lab,'R²'],   df_a.loc[lab,'R²']
    rrb, rra = df_b.loc[lab,'RMSE'], df_a.loc[lab,'RMSE']
    print(f'{lab:<14}  {rb:>7.4f}  {ra:>7.4f}  {ra-rb:>+7.4f}  '
          f'{rrb:>9.2f}  {rra:>9.2f}  {rra-rrb:>+7.2f}')
